# Notebook 2 — Reward Design and Policy Evaluation

**Module 3 — Applied Tabular RL**

## What this notebook covers
1. **Reward proxy effects** — how different reward functions lead to different (and incompatible) optimal policies
2. **Reward hacking** — an agent exploits a miscalibrated simulator rather than learning true dynamics
3. **Statistical evaluation** — paired comparison, four test statistics, effect size

## Same environment as Notebook 1
The toy market MDP (5 demand states, 3 price actions) is redefined here so this notebook is fully self-contained.

---
> **Core question:** Given that we cannot directly observe the true objective, how do we know whether a trained policy is actually better — and how confident can we be?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats

SEED = 42

# ── Environment constants ──────────────────────────────────────────────────
BASE_PRICE   = 10.0
ACTION_MULTS = np.array([0.85, 1.00, 1.15])
N_ACTIONS    = len(ACTION_MULTS)
N_STATES     = 5
A_NAMES      = ['discount', 'standard', 'premium']

TRUE_BASE_DEMAND = np.array([3.0,  8.0,  15.0, 25.0, 40.0])
TRUE_ELASTICITY  = np.array([-2.0, -1.5, -1.0, -0.7, -0.3])

_BINS = [5, 12, 20, 32]

def qty_bin(q: float) -> int:
    for i, t in enumerate(_BINS):
        if q < t:
            return i
    return len(_BINS)

def true_demand(s: int, mult: float) -> float:
    return float(TRUE_BASE_DEMAND[s] * (mult ** TRUE_ELASTICITY[s]))

def true_revenue_step(s: int, a: int) -> tuple[int, float]:
    mult = float(ACTION_MULTS[a])
    qty  = true_demand(s, mult)
    return qty_bin(qty), BASE_PRICE * mult * qty

# ── Fit simulator from data (replicates Notebook 1) ───────────────────────
def _generate_dataset(n: int = 1000, noise_std: float = 0.25, seed: int = SEED):
    rng     = np.random.default_rng(seed)
    states  = rng.integers(0, N_STATES, size=n)
    actions = rng.integers(0, N_ACTIONS, size=n)
    noise   = rng.normal(0.0, noise_std, size=n)
    qtys    = np.array([
        max(0.0, true_demand(int(states[i]), ACTION_MULTS[int(actions[i])]) * np.exp(noise[i]))
        for i in range(n)
    ])
    return states, actions, qtys

_states, _actions, _qtys = _generate_dataset()
_prices = BASE_PRICE * ACTION_MULTS[_actions]

_state_models: dict[int, LinearRegression] = {}
for s in range(N_STATES):
    mask = _states == s
    if mask.sum() < 5:
        continue
    m = LinearRegression().fit(np.log(_prices[mask]).reshape(-1, 1), np.log1p(_qtys[mask]))
    _state_models[s] = m

def sim_demand(s: int, mult: float) -> float:
    m = _state_models.get(s)
    if m is None:
        return true_demand(s, mult)
    log_q = float(m.predict([[np.log(BASE_PRICE * mult)]])[0])
    return max(0.0, float(np.expm1(np.clip(log_q, -10.0, 10.0))))

def sim_step(s: int, a: int) -> tuple[int, float]:
    mult = float(ACTION_MULTS[a])
    qty  = sim_demand(s, mult)
    return qty_bin(qty), BASE_PRICE * mult * qty

# ── Shared training function ───────────────────────────────────────────────
def train_q(
    step_fn,
    n_ep:      int   = 4000,
    horizon:   int   = 8,
    alpha:     float = 0.10,
    gamma:     float = 0.95,
    eps_start: float = 1.0,
    eps_end:   float = 0.05,
    seed:      int   = 0,
) -> tuple[np.ndarray, list[float]]:
    rng  = np.random.default_rng(seed)
    Q    = np.zeros((N_STATES, N_ACTIONS))
    rets = []
    for ep in range(n_ep):
        eps = eps_end + (eps_start - eps_end) * np.exp(-5.0 * ep / n_ep)
        s   = int(rng.integers(0, N_STATES))
        G   = 0.0
        for _ in range(horizon):
            a     = int(rng.integers(0, N_ACTIONS)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r = step_fn(s, a)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
            G += r
            s  = s2
        rets.append(G)
    return Q, rets

print("Setup complete.")

## Part 1 — Proxy Rewards: Revenue vs. Profit

The reward function defines what the agent optimizes. Using a **proxy** — a signal that correlates with but does not equal the true objective — can produce policies that are optimal for the proxy but suboptimal (or harmful) for the real goal.

**Experiment:** Train two policies on the true environment with different reward functions:

| Policy | Reward |
|--------|--------|
| `Q_rev` | Revenue = $p \cdot q$ |
| `Q_profit` | Profit = $(p - c) \cdot q$, where $c = 5.0$ is unit cost |

Both policies train on the *same* dynamics. Only the reward signal differs.

Then evaluate **both on true revenue** — so the comparison is apples-to-apples.

**Revenue-optimal** policy: $r \propto p^{1+e_s}$ — maximize over price multiplier.
- For $e_s < -1$ (elastic states): smaller price wins (volume dominates).
- For $e_s > -1$ (inelastic states): larger price wins (margin dominates).

**Profit-optimal** policy: $\pi \propto (p - c)^1 \cdot p^{e_s}$ — a different tradeoff between margin and volume.

In [ ]:
UNIT_COST = 5.0   # cost per unit produced/acquired

def true_profit_step(s: int, a: int) -> tuple[int, float]:
    """Same dynamics as true_revenue_step; reward = profit instead of revenue."""
    mult   = float(ACTION_MULTS[a])
    qty    = true_demand(s, mult)
    margin = BASE_PRICE * mult - UNIT_COST
    return qty_bin(qty), margin * qty

# Show the analytical optimal for each reward type
print("True optimal actions by reward function:")
print(f"{'s':>3}  {'e_s':>5}  {'rev-optimal':>14}  {'profit-optimal':>16}  same?")
for s in range(N_STATES):
    rev_vals    = [BASE_PRICE * ACTION_MULTS[a] * true_demand(s, ACTION_MULTS[a]) for a in range(N_ACTIONS)]
    profit_vals = [(BASE_PRICE * ACTION_MULTS[a] - UNIT_COST) * true_demand(s, ACTION_MULTS[a]) for a in range(N_ACTIONS)]
    r_opt = int(np.argmax(rev_vals))
    p_opt = int(np.argmax(profit_vals))
    same  = 'yes' if r_opt == p_opt else 'NO'
    print(f"{s:>3}  {TRUE_ELASTICITY[s]:>5.1f}  {A_NAMES[r_opt]:>14}  {A_NAMES[p_opt]:>16}  {same:>5}")

In [ ]:
Q_rev,    rev_rets    = train_q(true_revenue_step, seed=SEED)
Q_profit, profit_rets = train_q(true_profit_step,  seed=SEED)

pol_rev    = np.argmax(Q_rev,    axis=1)
pol_profit = np.argmax(Q_profit, axis=1)

print("Learned policies (from Q-tables trained on each reward):")
print(f"{'s':>3}  {'rev policy':>14}  {'profit policy':>16}  same?")
for s in range(N_STATES):
    same = 'yes' if pol_rev[s] == pol_profit[s] else 'NO'
    print(f"{s:>3}  {A_NAMES[pol_rev[s]]:>14}  {A_NAMES[pol_profit[s]]:>16}  {same:>5}")

# Learning curves
W = 200
fig, ax = plt.subplots(figsize=(7, 3))
for rets, label, c in [(rev_rets, 'Revenue reward', 'steelblue'), (profit_rets, 'Profit reward', 'seagreen')]:
    ax.plot(np.convolve(rets, np.ones(W) / W, mode='valid'), label=label, color=c)
ax.set_xlabel("Episode")
ax.set_ylabel(f"Return ({W}-ep avg)")
ax.set_title("Training Curves — Revenue vs Profit Reward")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate both policies on the TRUE revenue objective (fair comparison)
def eval_revenue(policy: np.ndarray, n: int = 3000, seed: int = 0) -> tuple[float, float]:
    """Returns (mean_revenue_policy, mean_revenue_baseline=standard)."""
    rng = np.random.default_rng(seed)
    pol_r, bl_r = [], []
    for _ in range(n):
        s = int(rng.integers(0, N_STATES))
        _, pr = true_revenue_step(s, int(policy[s]))
        _, br = true_revenue_step(s, 1)    # baseline: standard price
        pol_r.append(pr)
        bl_r.append(br)
    return float(np.mean(pol_r)), float(np.mean(bl_r))

def eval_profit(policy: np.ndarray, n: int = 3000, seed: int = 0) -> tuple[float, float]:
    """Returns (mean_profit_policy, mean_profit_baseline=standard)."""
    rng = np.random.default_rng(seed)
    pol_r, bl_r = [], []
    for _ in range(n):
        s = int(rng.integers(0, N_STATES))
        _, pr = true_profit_step(s, int(policy[s]))
        _, br = true_profit_step(s, 1)
        pol_r.append(pr)
        bl_r.append(br)
    return float(np.mean(pol_r)), float(np.mean(bl_r))

rev_pol_rev,    rev_bl = eval_revenue(pol_rev)
rev_pol_profit, _      = eval_revenue(pol_profit)
prof_pol_rev,   _      = eval_profit(pol_rev)
prof_pol_profit, prof_bl = eval_profit(pol_profit)

print("Evaluation (3,000 paired episodes):")
print(f"{'Policy':>18}  {'true revenue':>14}  {'true profit':>12}")
print(f"{'Baseline (std)':>18}  {rev_bl:>14.2f}  {prof_bl:>12.2f}")
print(f"{'Q_rev (revenue rwd)':>18}  {rev_pol_rev:>14.2f}  {prof_pol_rev:>12.2f}")
print(f"{'Q_profit (profit rwd)':>18}  {rev_pol_profit:>14.2f}  {prof_pol_profit:>12.2f}")
print()
print("Key insight: the profit-optimal policy accepts lower revenue to improve margin.")
print("Neither is universally 'better' — it depends on the actual business objective.")

## Part 2 — Reward Hacking

When the reward is computed through a **learned model**, the agent can exploit model inaccuracies instead of learning true dynamics.

**Setup:** We create a *miscalibrated* simulator that underestimates price elasticity:

$$\hat{e}_s = -0.2 \quad \forall s \quad \text{(true: } e_s \in [-2.0, -0.3]\text{)}$$

With $\hat{e} = -0.2$, the model predicts demand barely changes with price:

$$\hat{r}(a) = 10 \cdot \text{mult}^{1 + (-0.2)} = 10 \cdot b_s \cdot \text{mult}^{0.8}$$

Since $0.8 > 0$, the model always predicts: *higher price → more revenue*. The agent will learn to always pick the highest price — not because that is truly optimal, but because the **model says so**.

> This is reward hacking: the agent optimizes the proxy reward $\hat{r}$ while violating the true objective $r$.

In [ ]:
MISFIT_ELASTICITY = -0.2   # assumed elasticity — much less elastic than reality

def misfit_demand(s: int, mult: float) -> float:
    return float(TRUE_BASE_DEMAND[s] * (mult ** MISFIT_ELASTICITY))

def misfit_step(s: int, a: int) -> tuple[int, float]:
    mult = float(ACTION_MULTS[a])
    qty  = misfit_demand(s, mult)
    return qty_bin(qty), BASE_PRICE * mult * qty

# Show predicted vs true revenue: where does the model mislead the agent?
print("Misfit model revenue vs true revenue per (state, action):")
print(f"{'s':>3}  {'a':>3}  {'misfit_rev':>12}  {'true_rev':>10}  overestimates?")
for s in range(N_STATES):
    for a in range(N_ACTIONS):
        _, mr = misfit_step(s, a)
        _, tr = true_revenue_step(s, a)
        over  = 'YES' if mr > tr else 'no'
        print(f"{s:>3}  {a:>3}  {mr:>12.2f}  {tr:>10.2f}  {over}")

In [ ]:
Q_hacked, hacked_rets = train_q(misfit_step, seed=SEED)
pol_hacked = np.argmax(Q_hacked, axis=1)

TRUE_OPT = [
    int(np.argmax([BASE_PRICE * ACTION_MULTS[a] * true_demand(s, ACTION_MULTS[a])
                   for a in range(N_ACTIONS)]))
    for s in range(N_STATES)
]

print("Hacked policy vs true optimal:")
print(f"{'s':>3}  {'hacked':>12}  {'true opt':>12}  match?")
for s in range(N_STATES):
    chk = 'yes' if pol_hacked[s] == TRUE_OPT[s] else 'WRONG'
    print(f"{s:>3}  {A_NAMES[pol_hacked[s]]:>12}  {A_NAMES[TRUE_OPT[s]]:>12}  {chk}")

rev_hacked, _ = eval_revenue(pol_hacked)
rev_true_opt, _ = eval_revenue(np.array(TRUE_OPT))
print()
print(f"True revenue — hacked policy:   {rev_hacked:.2f}")
print(f"True revenue — true optimal:    {rev_true_opt:.2f}")
print(f"Baseline (standard):            {rev_bl:.2f}")

## Part 3 — Evaluation Methodology

To rigorously answer *"is policy $\pi$ better than baseline $\pi_b$?"* we need:

1. A **paired design** — evaluate both policies from the *same* initial state so initial-state effects cancel
2. **Multiple statistical tests** — each with different assumptions, together providing robust inference
3. **Effect size** — to distinguish practical significance from statistical significance

### The four tests

| Test | Null hypothesis | Assumption |
|------|----------------|------------|
| Paired t-test | $\mathbb{E}[d_i] = 0$ | $d_i$ approximately normal |
| Wilcoxon signed-rank | distribution of $d_i$ symmetric around 0 | Symmetric, no parametric form |
| Sign/binomial test | $P(d_i > 0) = 0.5$ | None (ignores magnitude) |
| Bootstrap CI | — | None (resampling) |

where $d_i = R_i^\pi - R_i^{\pi_b}$ is the per-episode return difference.

In [ ]:
def paired_evaluate(
    policy:          np.ndarray,
    baseline_action: int   = 1,    # standard price
    n_eval:          int   = 2000,
    seed:            int   = 0,
) -> dict:
    """Paired evaluation of policy vs baseline on the true environment."""
    rng      = np.random.default_rng(seed)
    pol_revs, bl_revs = [], []
    for _ in range(n_eval):
        s = int(rng.integers(0, N_STATES))
        _, pol_r = true_revenue_step(s, int(policy[s]))
        _, bl_r  = true_revenue_step(s, baseline_action)
        pol_revs.append(pol_r)
        bl_revs.append(bl_r)

    pol = np.asarray(pol_revs)
    bl  = np.asarray(bl_revs)
    d   = pol - bl

    t_stat, t_p   = stats.ttest_rel(pol, bl)
    try:
        w_stat, w_p = stats.wilcoxon(d)
    except ValueError:                      # all differences identical
        w_stat, w_p = float('nan'), float('nan')

    wins  = int((d > 0).sum())
    ties  = int((d == 0).sum())
    n_eff = n_eval - ties
    btest = stats.binomtest(wins, n_eff, p=0.5) if n_eff > 0 else None

    boot_rng   = np.random.default_rng(99)
    boot_means = [
        boot_rng.choice(d, size=len(d), replace=True).mean()
        for _ in range(2000)
    ]
    ci = (float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5)))

    return {
        'pol_mean':     float(pol.mean()),
        'bl_mean':      float(bl.mean()),
        'lift_pct':     float(100.0 * d.mean() / (bl.mean() + 1e-12)),
        'paired_t':     (float(t_stat), float(t_p)),
        'wilcoxon':     (float(w_stat), float(w_p)),
        'sign_test':    {'wins': wins, 'n_eff': n_eff,
                         'p': float(btest.pvalue) if btest else float('nan')},
        'bootstrap_ci': ci,
        'cohens_d':     float(d.mean() / (d.std(ddof=1) + 1e-12)),
    }

def print_eval(name: str, res: dict) -> None:
    t, tp = res['paired_t']
    w, wp = res['wilcoxon']
    st    = res['sign_test']
    ci    = res['bootstrap_ci']
    print("-" * 58)
    print(f"Policy: {name}")
    print(f"  Mean — policy: {res['pol_mean']:.2f}   baseline: {res['bl_mean']:.2f}")
    print(f"  Lift:  {res['lift_pct']:+.2f}%   Cohen's d: {res['cohens_d']:+.3f}")
    print(f"  Paired t-test:  t={t:.3f},  p={tp:.4f}")
    print(f"  Wilcoxon:       W={w:.1f},  p={wp:.4f}")
    print(f"  Sign test:      wins={st['wins']}/{st['n_eff']},  p={st['p']:.4f}")
    print(f"  Bootstrap 95%:  [{ci[0]:+.2f}, {ci[1]:+.2f}]")

In [ ]:
# Evaluate all four policies
res_rev    = paired_evaluate(pol_rev)
res_profit = paired_evaluate(pol_profit)
res_hacked = paired_evaluate(pol_hacked)
res_true   = paired_evaluate(np.array(TRUE_OPT))

for name, res in [
    ("Revenue policy (Q_rev)",    res_rev),
    ("Profit policy (Q_profit)",  res_profit),
    ("Reward hacking (Q_hacked)", res_hacked),
    ("True optimal (analytical)", res_true),
]:
    print_eval(name, res)
print("-" * 58)

In [ ]:
labels = ['Q_rev', 'Q_profit', 'Q_hacked', 'True opt']
results = [res_rev, res_profit, res_hacked, res_true]
lifts   = [r['lift_pct']  for r in results]
d_vals  = [r['cohens_d']  for r in results]
ci_los  = [r['bootstrap_ci'][0] for r in results]
ci_his  = [r['bootstrap_ci'][1] for r in results]
colors  = ['steelblue', 'seagreen', 'tomato', 'goldenrod']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Revenue lift with bootstrap CI (clamp error bar extents to >=0)
err_below = [max(0.0, lifts[i] - ci_los[i]) for i in range(len(lifts))]
err_above = [max(0.0, ci_his[i] - lifts[i]) for i in range(len(lifts))]
ax1.bar(labels, lifts, color=colors, alpha=0.85)
ax1.errorbar(
    range(len(labels)), lifts,
    yerr=[err_below, err_above],
    fmt='none', color='black', capsize=5, linewidth=1.5,
)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_ylabel("Revenue lift vs baseline (%)")
ax1.set_title("Revenue Lift (with 95% Bootstrap CI)")

# Effect size
ax2.bar(labels, d_vals, color=colors, alpha=0.85)
for thresh, lbl in [(0.2, 'small'), (0.5, 'medium'), (0.8, 'large')]:
    ax2.axhline(thresh, color='gray', linewidth=0.7, linestyle=':')
    ax2.text(3.6, thresh + 0.02, lbl, fontsize=8, color='gray')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_ylabel("Cohen's d")
ax2.set_title("Effect Size")

plt.suptitle("Policy Comparison — Paired Evaluation on True Environment", y=1.02)
plt.tight_layout()
plt.show()

# Distribution of per-episode differences for each policy
policies_map = {
    'Q_rev':     pol_rev,
    'Q_profit':  pol_profit,
    'Q_hacked':  pol_hacked,
    'True opt':  np.array(TRUE_OPT),
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (name, color) in zip(axes, zip(labels, colors)):
    pol = policies_map[name]
    rng = np.random.default_rng(0)
    diffs = []
    for _ in range(2000):
        s = int(rng.integers(0, N_STATES))
        _, pr = true_revenue_step(s, int(pol[s]))
        _, br = true_revenue_step(s, 1)
        diffs.append(pr - br)
    ax.hist(diffs, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("d_i = r_pol - r_base")
plt.suptitle("Per-Episode Difference Distribution", y=1.02)
plt.tight_layout()
plt.show()

## Summary

### Part 1 — Proxy Rewards
- Revenue-optimal and profit-optimal policies differ at states where the elasticity makes the price-volume tradeoff non-trivial.
- Optimizing the wrong proxy produces a policy that is suboptimal on the true objective.
- The gap depends on how misaligned the proxy is — here it is non-trivial only at low-demand states.

### Part 2 — Reward Hacking
- A miscalibrated simulator (underestimated elasticity) causes the agent to always pick the highest price.
- Evaluated on the true environment, the hacked policy **underperforms the baseline** at cold demand states.
- Diagnostic: if the trained policy selects the same extreme action for *all* states regardless of context, it is likely hacking the model rather than learning true structure.

### Part 3 — Evaluation
- Paired comparison eliminates initial-state confounding.
- Wilcoxon + bootstrap CI are more reliable than the t-test alone when the return distribution is heavy-tailed or skewed.
- Statistical significance alone is not enough — Cohen's d and the bootstrap CI communicate practical effect size.

---

| Policy | True revenue lift | Cohen's d | Interpretation |
|--------|------------------|-----------|----------------|
| Q_rev  | positive | moderate | Correct proxy, good result |
| Q_profit | smaller positive | smaller | Optimizes different objective |
| Q_hacked | near zero or negative | near zero | Exploits model, not reality |
| True optimal | highest | highest | Upper bound |

---
**Module 3 complete.** Next: Module 4 — Deep RL (DQN) replaces the Q-table with a neural network to handle continuous state spaces.